# Model training for splice-site-predictor using prepared data

In [5]:
# install pytorch
# !pip install torch
!pip install scikit-learn

  Obtaining dependency information for scikit-learn from https://files.pythonhosted.org/packages/8d/da/4810a28e473185429e45a57eebcc91fc991b33d889cc0676063e671db03d/scikit_learn-1.9.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for scipy>=1.10.0 from https://files.pythonhosted.org/packages/09/7d/af933f0f6e0767995b4e2d705a0665e454d1c19402aa7e895de3951ebb04/scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 499.2 kB/s eta 0:00:00m eta 0:00:01:--
  Obtaining dependency information for joblib>=1.4.0 from https://files.pythonhosted.org/packages/7b/91/984aca2ec129e2757d1e4e3c81c3fcda9d0f85b74670a094cc443d9ee949/joblib-1.5.3-py3-none-any.whl.metadata
  Obtaining dependency information for narwhals>=2.0.1 from https://files.pythonhosted.org/packages/48/ca/36339329c4604adbcc99c899b7eb1ce1a555c499b6a6860757dc9bfed36d/narwhals-2.22.1-py3-none-any.whl.

In [7]:
# load datset (combine positives and negatives)

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Load one-hot encoded files
positives = np.load('data/positives.npy')   # shape: (150000, 400)
negatives = np.load('data/negatives.npy')    # shape: (150000, 400)

# Create labels
# 1 = real splice donor site, 0 = non-donor GT dinucleotide
pos_labels = np.ones(len(positives),  dtype=np.float32)  # 150000 ones
neg_labels = np.zeros(len(negatives), dtype=np.float32)  # 150000 zeros

# Combine into one dataset
X = np.concatenate([positives, negatives], axis=0)  # shape: (300000, 400)
y = np.concatenate([pos_labels, neg_labels], axis=0) # shape: (300000,)

print(f"X shape: {X.shape}")  # (300000, 400)
print(f"y shape: {y.shape}")  # (300000,)
print(f"Class balance: {y.mean():.2f}")  # should be 0.50 → balanced

X shape: (310028, 400)
y shape: (310028,)
Class balance: 0.50


In [8]:
# Split into train / validation / test set

# Seperate the test set
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=0.10,       # 10% → 30,000 samples for testing
    random_state=42,      # reproducibility
    stratify=y            # keeps 50/50 class balance in each split
)

# Split remaining data into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.111,      # ~10% of total --> 30,000 validation samples
    random_state=42,
    stratify=y_trainval
)

# Check sizes
print(f"Train:      {X_train.shape[0]:>7,} samples")  # ~240,000
print(f"Validation: {X_val.shape[0]:>7,} samples")    # ~30,000
print(f"Test:       {X_test.shape[0]:>7,} samples")   # ~30,000

Train:      248,053 samples
Validation:  30,972 samples
Test:        31,003 samples


In [9]:
# create Pytorch data set

class SpliceSiteDataset(Dataset):
    """
    A PyTorch Dataset for splice donor site classification.
    Stores one-hot encoded sequences and their binary labels.
    """
    def __init__(self, X, y):
        # Convert numpy arrays to PyTorch tensors once (efficient)
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        """Returns total number of samples — used by DataLoader."""
        return len(self.X)

    def __getitem__(self, idx):
        """Returns one sample by index — called by DataLoader."""
        return self.X[idx], self.y[idx]


# Set up the 3 datasets
train_dataset = SpliceSiteDataset(X_train, y_train)
val_dataset   = SpliceSiteDataset(X_val,   y_val)
test_dataset  = SpliceSiteDataset(X_test,  y_test)

# Sanity check
sample_X, sample_y = train_dataset[0]
print(f"One sample — X shape: {sample_X.shape}, label: {sample_y}")
# X shape: torch.Size([400]), label: tensor(1.)

One sample — X shape: torch.Size([400]), label: 1.0


In [20]:
# Create data loaders

BATCH_SIZE = 256  # number of samples processed together in one forward pass

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,       # ← shuffle ONLY training data
    num_workers=0,      # parallel data loading (use 0 on Windows if issues)
    pin_memory=False     # True for faster GPU transfer
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,      # ← no shuffling for val/test (order doesn't matter)
    num_workers=0,
    pin_memory=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

# --- Inspect one batch ---
X_batch, y_batch = next(iter(train_loader))
print(f"Batch X shape: {X_batch.shape}")  # torch.Size([256, 400])
print(f"Batch y shape: {y_batch.shape}")  # torch.Size([256])

Batch X shape: torch.Size([256, 400])
Batch y shape: torch.Size([256])


In [21]:
# Reshape for CNN

class SpliceSiteDataset(Dataset):
    def __init__(self, X, y, reshape_for_cnn=True):
        X_tensor = torch.tensor(X, dtype=torch.float32)
        
        if reshape_for_cnn:
            # (N, 400) --> (N, 100, 4) --> (N, 4, 100)
            # Encoding order: [A,C,G,T] repeated per position
            X_tensor = X_tensor.view(-1, 100, 4).permute(0, 2, 1)
        
        self.X = X_tensor
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Shape check:
# Flat MLP input:  torch.Size([256, 400])
# CNN input:       torch.Size([256, 4, 100]) # batch, channels, sequence_length

In [13]:
# install matplotlib if necessary
!pip install matplotlib

  Obtaining dependency information for matplotlib from https://files.pythonhosted.org/packages/53/f4/f0b4f9ba7ec14a7af8151f3ad71ecfe3561e6ba38cfab1db3681ba4ca112/matplotlib-3.11.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 1.3 MB/s eta 0:00:001.9 MB/s eta 0:00:01
  Obtaining dependency information for contourpy>=1.0.1 from https://files.pythonhosted.org/packages/5f/4b/6157f24ca425b89fe2eb7e7be642375711ab671135be21e6faa100f7448c/contourpy-1.3.3-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for cycler>=0.10 from https://files.pythonhosted.org/packages/e7/05/c19819d5e3d95294a6f5947fb9b9629efb316b96de511b418c53d245aae6/cycler-0.12.1-py3-none-any.whl.metadata
  Obtaining dependency information for fonttools>=4.22.0 from https://files.pythonhosted.org/packages/0b/43/a81f20050a3115b57d62c8e781446949512eac36690dc384ccea65ff4cc1/fonttools-4.63.0-cp311-cp3

In [22]:
# Setup
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Use GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [23]:
# Define the model

class SpliceSiteCNN(nn.Module):
    """
    1D CNN for binary splice donor site classification.
    Input shape:  (batch_size, 4, 100)
                          channels  sequence length
                         (A,C,G,T)  (100 bp window)
    Output shape: (batch_size,) — one score per sample
    """

    def __init__(self):
        super(SpliceSiteCNN, self).__init__()

        # Three convolutional blocks
        # Each block: Conv --> BatchNorm --> ReLU --> MaxPool
        # Channels increase (4-->32-->64-->128): detect increasingly complex patterns
        # MaxPool halves the sequence length each time

        self.conv_block1 = nn.Sequential(
            nn.Conv1d(in_channels=4,   # 4 input channels (A, C, G, T)
                      out_channels=32, # learn 32 different motif detectors
                      kernel_size=9,   # each filter sees 9 bp at a time
                      padding='same'), # keep sequence length unchanged
            nn.BatchNorm1d(32),        # normalise activations --> stable training
            nn.ReLU(),                 # non-linearity --> learn complex patterns
            nn.MaxPool1d(kernel_size=2) # 100 --> 50: keep strongest activations
        )

        self.conv_block2 = nn.Sequential(
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=8, padding='same'),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)  # 50 --> 25
        )

        self.conv_block3 = nn.Sequential(
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=8, padding='same'),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)  # 25 --> 12
        )

        # Fully connected classifier
        # Takes the flattened conv output and maps it to a single score
        self.classifier = nn.Sequential(
            nn.Flatten(),              # (batch, 128, 12) --> (batch, 1536)
            nn.Linear(128 * 12, 256), # 1536 --> 256
            nn.ReLU(),
            nn.Dropout(0.5),           # randomly zero 50% of neurons --> prevents overfitting
            nn.Linear(256, 1)          # 256 --> 1 score per sample
            # Sigmoid is applied internally
        )

    def forward(self, x):
        """Defines how data flows through the network."""
        x = self.conv_block1(x)   # (batch, 4,  100) --> (batch, 32, 50)
        x = self.conv_block2(x)   # (batch, 32,  50) --> (batch, 64, 25)
        x = self.conv_block3(x)   # (batch, 64,  25) --> (batch, 128, 12)
        x = self.classifier(x)    # (batch, 128, 12) --> (batch, 1)
        return x.squeeze(1)       # (batch, 1) --> (batch,) — remove extra dimension


# Move to device
model = SpliceSiteCNN().to(device)
print(model)

# Quick shape check
dummy_input = torch.zeros(9, 4, 100).to(device)  # fake batch of 8
dummy_output = model(dummy_input)
print(f"\nInput shape:  {dummy_input.shape}")   # torch.Size([8, 4, 100])
print(f"Output shape: {dummy_output.shape}")    # torch.Size([8])

SpliceSiteCNN(
  (conv_block1): Sequential(
    (0): Conv1d(4, 32, kernel_size=(9,), stride=(1,), padding=same)
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block2): Sequential(
    (0): Conv1d(32, 64, kernel_size=(8,), stride=(1,), padding=same)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block3): Sequential(
    (0): Conv1d(64, 128, kernel_size=(8,), stride=(1,), padding=same)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1):

In [24]:
# Loss function
criterion = nn.BCEWithLogitsLoss()
# BCEWithLogitsLoss = sigmoid + binary cross-entropy combined

# Optimiser
optimizer = torch.optim.Adam(
    model.parameters(),  # all weights and biases to optimise
    lr=0.001             # learning rate
)
# Adam: adapts the learning rate per parameter

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',      # we want to MINIMISE validation loss
    patience=3,      # wait 3 epochs with no improvement before reducing lr
    factor=0.5       # multiply lr by 0.5 when triggered
)
# If training stalls, automatically lowers the learning rate

In [25]:
# Training and validation functions
def train_one_epoch(model, loader, criterion, optimizer, device):
    """
    Runs one full pass through the training data.
    Returns the average loss across all batches.
    """
    model.train()  # IMPORTANT: activates dropout and batch norm training behaviour
    running_loss = 0.0

    for X_batch, y_batch in loader:

        # Move batch to GPU/CPU
        X_batch = X_batch.to(device)   # (batch, 400)
        y_batch = y_batch.to(device)   # (batch,)

        # Reshape for CNN
        # (batch, 400) --> (batch, 100, 4) --> (batch, 4, 100)
        X_batch = X_batch.view(-1, 100, 4).permute(0, 2, 1)

        # Step 1: Clear gradients from previous batch
        optimizer.zero_grad()
        # Gradients accumulate by default in PyTorch —-> reset each batch

        # Step 2: Forward pass
        predictions = model(X_batch)   # (batch,) — raw scores (logits)

        # Step 3: Compute loss
        loss = criterion(predictions, y_batch)
        # Compares model predictions to true labels (0 or 1)

        # Step 4: Backward pass
        loss.backward()
        # Computes gradients: how much each weight contributed to the loss

        # Step 5: Update weights
        optimizer.step()
        # Adjusts all weights in the direction that reduces loss

        running_loss += loss.item()  # .item() extracts Python float from tensor

    avg_loss = running_loss / len(loader)  # average over all batches
    return avg_loss


def validate(model, loader, criterion, device):
    """
    Runs one full pass through the validation data.
    Returns the average loss, NO weight updates.
    """
    model.eval()   # IMPORTANT: disables dropout, uses running stats for batch norm
    running_loss = 0.0

    with torch.no_grad():
        # torch.no_grad(): skip building the computation graph
        # uses less memory, runs faster, no gradients for evaluation

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            X_batch = X_batch.view(-1, 100, 4).permute(0, 2, 1)

            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)

            running_loss += loss.item()

    avg_loss = running_loss / len(loader)
    return avg_loss

In [26]:
# Full training loop

NUM_EPOCHS   = 30
train_losses = []
val_losses   = []
best_val_loss = float('inf')  # track best model

print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'LR':>10}")
print("-" * 45)

for epoch in range(NUM_EPOCHS):

    # Train and validate
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss   = validate(model, val_loader, criterion, device)

    # Record losses
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # Step the scheduler
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        note = " ← best"
    else:
        note = ""

    # Print progress
    print(f"{epoch+1:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | {current_lr:>10.6f}{note}")

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

 Epoch | Train Loss |   Val Loss |         LR
---------------------------------------------


KeyboardInterrupt: 